## 1. Import Requirements

# Master Shear-Wave Splitting Workflow for Axial Seamount

This notebook provides a complete, clean workflow from raw earthquake catalog and waveform data to shear-wave splitting analysis results. The workflow follows proper sequencing and includes all necessary quality control measures. 

Instead of using catalog from Wilcock and Zhang or ML DD, we use the nlloc file for all stations from Christian's results.

## Workflow Overview

1. **Data Loading & Initial Setup** - Load earthquake catalog and station metadata
2. **Extended Time Window Creation** - Create proper time windows for waveform retrieval
3. **Waveform Data Retrieval** - Download seismic data with extended windows
4. **Quality Control Filters** - P-wave rectilinearity, SNR, and incidence angle filtering
5. **Geometric Calculations** - Back-azimuth and distance calculations
6. **Shear-Wave Splitting Analysis** - Dynamic parameter estimation and SWSPy analysis
7. **Results Processing & Visualization** - Compile and visualize splitting parameters

## Key Improvements
- Extended catalog creation moved to proper early position
- Updated P-wave polarization analysis for true incidence angles
- Integrated SNR calculations with proper S-wave timing
- Clean separation of quality control steps

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import obspy
from obspy.core.utcdatetime import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.core.event import read_events
import os
import sys
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Add local swspy directory to path (before other imports)
swspy_local_path = os.path.abspath('../swspy')
if swspy_local_path not in sys.path:
    sys.path.insert(0, swspy_local_path)

# Import swspy from local directory
import swspy

# Add scripts directory to path for custom modules
sys.path.append('.')
from get_all_traces import get_station_traces_batch
from splitting_functions import *
from teanby_clustering import *

# Set up plotting
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully")
print(f"ObsPy version: {obspy.__version__}")
print(f"SWSPy available: {'Yes' if 'swspy' in sys.modules else 'No'}")
print(f"SWSPy location: {swspy.__file__}")

In [ ]:
# Load 2018 earthquake test catalog - ML DD
#catalog = pd.read_csv('2018_eq_catalog.csv')

# Load Baillard nonlinloc catalog
catalog = pd.read_csv('AXIAL.PHASE.FINAL_3D_V2.csv')

#Load station information from Christian's data
stations_file = '../data/stations_axial.llz'
#Read llz file - reads like a text file with space delimiter
stations_df = pd.read_csv(stations_file, delim_whitespace=True, header=None, names=['Longitude (°W)', 'Latitude (°N)', 'Elevation (m)', 'Station ID'])
# Convert elevation column to m from km
stations_df['Elevation (m)'] = stations_df['Elevation (m)']*1000

print(f"Stations in catalog: {catalog['station'].value_counts()}")

In [ ]:
# Filter catalog for AXAS2 station only
#axas2_catalog = catalog[catalog['station'] == 'OOAXAS2'].copy()

axas2_catalog = catalog[catalog['station'] == 'AXAS2'].copy()
#axec2_catalog = catalog[catalog['station'] == 'AXEC2'].copy()

# Reset index to ensure clean indexing
axas2_catalog = axas2_catalog.reset_index(drop=True)
#axec2_catalog = axec2_catalog.reset_index(drop=True)

# Pick all events from April and May, 2015
axas2_catalog['datetime'] = pd.to_datetime(axas2_catalog['datetime'])
axas2_catalog = axas2_catalog[(axas2_catalog['datetime'] >= '2015-04-20') & (axas2_catalog['datetime'] < '2015-04-28')].copy()

#axec2_catalog['datetime'] = pd.to_datetime(axec2_catalog['datetime'])
#axec2_catalog = axec2_catalog[(axec2_catalog['datetime'] >= '2015-04-20') & (axec2_catalog['datetime'] < '2015-04-28')].copy()

# Select first 100 events for testing
#test_catalog_100 = axas2_catalog.head(100).copy()
print(f"Total AXAS2 events in catalog: {len(axas2_catalog)}")
#print(f"Total AXEC2 events in catalog: {len(axec2_catalog)}")

#print(f"Test catalog created with first {len(test_catalog_100)} AXAS2 events")
print(f"\nDate range of test catalog:")
print(f"Start: {axas2_catalog['datetime'].min()}")
print(f"End: {axas2_catalog['datetime'].max()}")
#print(f"Start: {axec2_catalog['datetime'].min()}")
#print(f"End: {axec2_catalog['datetime'].max()}")

display(axas2_catalog)
#display(axec2_catalog)

In [ ]:
# Downsample catalog by a factor of 4 - take every fourth event
#test_catalog = axas2_catalog.iloc[::4].copy()

test_catalog = axas2_catalog.copy()
#test_catalog = axec2_catalog.copy()

print(f"Test catalog created with every 4th AXAS2 event, total {len(test_catalog)} events")
print(f"\nDate range of test catalog:")
print(f"Start: {test_catalog['datetime'].min()}")
print(f"End: {test_catalog['datetime'].max()}")

In [ ]:
# First, reformat the datetime strings to add 'T' separator
test_catalog['p_time'] = test_catalog['p_time'].str.replace(' ', 'T', regex=False)
test_catalog['s_time'] = test_catalog['s_time'].str.replace(' ', 'T', regex=False)
test_catalog['datetime'] = test_catalog['datetime'].astype(str).str.replace(' ', 'T', regex=False)

# Now convert to pandas Timestamp with UTC timezone
test_catalog['p_time'] = pd.to_datetime(test_catalog['p_time'], utc=True, format='ISO8601')
test_catalog['s_time'] = pd.to_datetime(test_catalog['s_time'], utc=True, format='ISO8601')
test_catalog['datetime'] = pd.to_datetime(test_catalog['datetime'], utc=True, format='ISO8601')

# Convert to UTCDateTime
test_catalog['p_time'] = test_catalog['p_time'].apply(lambda x: UTCDateTime(x))
test_catalog['s_time'] = test_catalog['s_time'].apply(lambda x: UTCDateTime(x))
test_catalog['datetime'] = test_catalog['datetime'].apply(lambda x: UTCDateTime(x))

print("Successfully converted to UTCDateTime")
print(f"Sample p_time: {test_catalog['p_time'].iloc[0]}")

In [ ]:
catalog = test_catalog.copy()

## 3. Extended Time Window Creation

This step creates extended time windows for waveform retrieval. This is critical for proper analysis and must happen early in the workflow, before any quality control that depends on waveform data.

In [ ]:
# Create extended time windows for proper waveform analysis
print("Creating extended time windows for waveform retrieval...")

# Apply extended windowing
extended_catalog = create_extended_catalog(catalog, pre_p_time=1.0, post_s_time=2.0)

print(f"Extended catalog created with {len(extended_catalog)} events")
print(f"Time windows: {extended_catalog['total_duration'].iloc[0]} seconds total")
print(f"Pre-event: {extended_catalog['pre_p_sec'].iloc[0]}s, Post-event: {extended_catalog['post_s_sec'].iloc[0]}s")
# Display sample of extended timing
print("\nSample timing windows:")
sample_cols = ['id', 'datetime', 'starttime', 'endtime', 'total_duration']
display(extended_catalog[sample_cols].head())

In [ ]:
# Remove leading 'OO' from station names
extended_catalog['station'] = extended_catalog['station'].str.replace('OO', '', regex=False)

In [ ]:
display(extended_catalog)

## 4. Waveform Data Retrieval

This section retrieves seismic waveform data using the extended time windows. We'll load the existing trace data and organize it for processing.

In [ ]:
test_catalog = extended_catalog

In [ ]:
# Replace catalog id with index
test_catalog['id'] = test_catalog.index

test_catalog['mag'] = 0.0

In [ ]:
# Retrieve waveforms for all events in the test catalog using get_all_traces function
#print("Retrieving waveforms for all events in the test catalog...")
#waveforms = get_station_traces_batch(test_catalog, 'axial_nonlinloc_april_20_28', 'starttime', 'endtime', 'station', batch_size=100)

In [ ]:
# Load waveforms from mseed file with obspy
waveforms_file = 'axial_nonlinloc_april_20_28.mseed'
waveforms = obspy.read(waveforms_file)

In [ ]:
# Associate waveforms with events in the catalog
print("Organizing waveforms by events...")
waveform_dict = organize_stream_by_events(waveforms, test_catalog)

In [ ]:
# Organize waveforms by event ID
print("Organizing waveforms by event ID...")
organized_waveforms = organize_waveform_data(waveform_dict, test_catalog)

In [ ]:
# Format s_arrival_time and p_arrival_time as difference between arrival times and origin time
print("Formatting s_arrival_time and p_arrival_time as differences from origin time...")
for eid in organized_waveforms.keys():
    organized_waveforms[eid]['s_arrival_time'] = (UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 's_time'].values[0]) - 
                                                  UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'datetime'].values[0]))
    organized_waveforms[eid]['p_arrival_time'] = (UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'p_time'].values[0]) - 
                                                  UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'datetime'].values[0]))

In [ ]:
# For all traces in organized_waveforms, taper and filter in-place
print("Tapering and filtering all traces in organized_waveforms...")
events_to_remove = []
try:
    for eid in organized_waveforms.keys():
        for tr in organized_waveforms[eid]['traces']:
            tr.detrend("linear") # to avoid weird start and end amplitudes
            tr.taper(max_percentage=0.05, type='hann')
            tr.filter('bandpass', freqmin=5.0, freqmax=40.0)
except Exception as e:
    print(f"Error during waveform processing: {e}")
    if type(organized_waveforms[eid]['traces']) == type(None):
        events_to_remove.append(eid)
    print(f"Events with issues: {events_to_remove}")

print("Waveform retrieval and organization complete.")

In [ ]:
# Remove duplicate traces from organized_waveforms
print("Checking for and removing duplicate traces in organized_waveforms...")

for eid in organized_waveforms.keys():
    # Get the stream for this event
    st = organized_waveforms[eid]['traces']
    
    # Check if there are duplicates
    try:
        if len(st) > 3:
            print(f"Event {eid}: Found {len(st)} traces (expected 3)")
            
            # Create a new stream with unique traces based on channel code
            unique_traces = {}
            for tr in st:
                channel = tr.stats.channel
                # Keep the first occurrence of each channel
                if channel not in unique_traces:
                    unique_traces[channel] = tr
            
            # Replace the stream with deduplicated traces
            organized_waveforms[eid]['traces'] = obspy.Stream(traces=list(unique_traces.values()))
            print(f"  Reduced to {len(organized_waveforms[eid]['traces'])} unique traces")
    except Exception as e:
        print(f"Error processing event {eid}: {e}")
        organized_waveforms[eid]['traces'] = st[:3]  # Fallback to first 3 traces if error occurs

# Verify the results
print("\nVerification of trace counts after deduplication:")
trace_counts = {}
for eid in organized_waveforms.keys():
    count = len(organized_waveforms[eid]['traces'])
    trace_counts[count] = trace_counts.get(count, 0) + 1

print(f"Events with 3 traces: {trace_counts.get(3, 0)}")
if any(k != 3 for k in trace_counts.keys()):
    print("Events with unexpected trace counts:")
    for count, num_events in trace_counts.items():
        if count != 3:
            print(f"  {num_events} events with {count} traces")
else:
    print("All events have exactly 3 traces (E, N, Z)")

In [ ]:
# Remove events that do not have exactly 3 traces
print("\nRemoving events that do not have exactly 3 traces...")
events_to_remove = []
for eid in organized_waveforms.keys():
    if len(organized_waveforms[eid]['traces']) != 3:
        events_to_remove.append(eid)

    # also remove events with any trace that has zero length (indicating a retrieval issue) or empty traces
    for tr in organized_waveforms[eid]['traces']:
        if tr.stats.npts == 0:
            print(f"Event {eid} has a trace with zero length, marking for removal")
            events_to_remove.append(eid)
            break

for eid in events_to_remove:
    del organized_waveforms[eid]

In [ ]:
len(organized_waveforms)

## 5. Quality Control Pipeline

This section implements comprehensive quality control measures including P-wave rectilinearity analysis, signal-to-noise ratio calculations, and incidence angle filtering.

In [ ]:
# Define quality control thresholds
QC_THRESHOLDS = {
    'min_snr': 2.0,           # Minimum S-wave signal-to-noise ratio
    'min_rectilinearity': 0.7, # Minimum P-wave rectilinearity
    'max_incidence': 30.0,     # Maximum incidence angle (degrees)
    'min_magnitude': 0.0,      # Minimum event magnitude
}

print("Quality control functions loaded successfully")
print(f"QC Thresholds: {QC_THRESHOLDS}")

In [ ]:
# Check that all traces for same event have same length, and remove events that do not meet this criterion
print("Checking that all traces for the same event have the same length...")
events_to_remove = []
for eid in organized_waveforms.keys():
    trace_lengths = [tr.stats.npts for tr in organized_waveforms[eid]['traces']]
    if len(set(trace_lengths)) != 1:
        print(f"Event {eid} has traces of different lengths: {trace_lengths}, marking for removal")
        events_to_remove.append(eid)

In [ ]:
# Check if any traces are length zero, and if so mark those events for removal
print("Checking for traces with zero length...")
events_to_remove = []
for eid in organized_waveforms.keys():
    for tr in organized_waveforms[eid]['traces']:
        if tr.stats.npts == 0:
            print(f"Event {eid} has a trace with zero length, marking for removal")
            events_to_remove.append(eid)
            break

In [ ]:
# Calculate quality control metrics for organized waveforms
print("Calculating quality control metrics for organized waveforms...")

# 1. Calculate S-wave SNR
organized_waveforms = calculate_snr_for_organized_waveforms(organized_waveforms)

# 2. Calculate geographic back-azimuth, for coordinate rotation later
organized_waveforms = calculate_back_azimuth_for_organized_waveforms(organized_waveforms, stations_df)

# 3. Calculate incidence angle
organized_waveforms = calculate_incidence_angle_eigenvalue_jurkevics_for_organized_waveforms(organized_waveforms, p_arrival_variable='p_arrival_time', analysis_window=0.12)

#4. Calculate P-wave rectilinearity
organized_waveforms = calculate_rectilinearity_jurkevics_for_organized_waveforms(organized_waveforms, p_arrival_variable='p_arrival_time', analysis_window=0.12)

In [ ]:
# Create a dataframe with quality control metrics for the test catalog events
qc_metrics_df = pd.DataFrame({
    'event_id': list(organized_waveforms.keys()),
    's_time': [organized_waveforms[eid]['s_arrival_time'] for eid in organized_waveforms.keys()],
    'station': [organized_waveforms[eid]['station'] for eid in organized_waveforms.keys()],
    'back_azimuth': [organized_waveforms[eid]['back_azimuth'] for eid in organized_waveforms.keys()],
    'snr_horizontal': [organized_waveforms[eid]['snr_horizontal'] for eid in organized_waveforms.keys()],
    'incidence': [organized_waveforms[eid]['incidence_eigenvalue_jurkevics'] for eid in organized_waveforms.keys()],
    'rectilinearity': [organized_waveforms[eid]['rectilinearity_jurkevics'] for eid in organized_waveforms.keys()],
    'latitude': [organized_waveforms[eid]['latitude'] for eid in organized_waveforms.keys()],
    'longitude': [organized_waveforms[eid]['longitude'] for eid in organized_waveforms.keys()],
    'depth': [organized_waveforms[eid]['depth'] for eid in organized_waveforms.keys()],
    'origin_time' : [organized_waveforms[eid]['origin_time'] for eid in organized_waveforms.keys()]
})

print(f"Quality control metrics dataframe created with {len(qc_metrics_df)} events")
display(qc_metrics_df)

In [ ]:
qc_metrics_df['s_time'] = qc_metrics_df['origin_time'] + qc_metrics_df['s_time']

In [ ]:
display(qc_metrics_df)

In [ ]:
# Define passing_waveforms as those that meet all QC thresholds
passing_waveforms = apply_quality_control(organized_waveforms, QC_THRESHOLDS)

In [ ]:
# Save the passing_waveforms
metadata_df = save_passing_waveforms(passing_waveforms, output_dir='passing_waveforms_data')
display(metadata_df.head())

In [ ]:
# Reload the data
passing_waveforms_reloaded = load_passing_waveforms(output_dir='passing_waveforms_data')

# Verify the reload worked correctly
print(f"Reloaded events: {len(passing_waveforms_reloaded)}")
print(f"\nSample reloaded event (ID: {list(passing_waveforms_reloaded.keys())[0]}):")
sample_event = passing_waveforms_reloaded[list(passing_waveforms_reloaded.keys())[0]]
print(f"  Station: {sample_event['station']}")
print(f"  Origin time: {sample_event['origin_time']}")
print(f"  Number of traces: {len(sample_event['traces'])}")
print(f"  Back azimuth: {sample_event['back_azimuth']:.2f}°")

## 6. Shear-Wave Splitting Analysis

This section implements the core shear-wave splitting analysis using SWSPy with dynamic parameter estimation and comprehensive quality assessment.

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
#print("Performing shear-wave splitting analysis using SWSPy method...")
#results_baillard = perform_splitting_on_organized_waveforms(passing_waveforms, use_dynamic_params=False, mode='baillard', plot_results=False)

In [ ]:
# Save results_baillard dictionary to CSV file for later analysis
#results_df = pd.DataFrame.from_dict(results_baillard, orient='index')
#results_df.to_csv('results_baillard_nonlinloc_april_20_28.csv')

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
print("Performing shear-wave splitting analysis using SWSPy method...")
results_swspy = perform_splitting_on_organized_waveforms(passing_waveforms, first_window_start=3, last_window_start=0, 
                                                         first_window_end=1.5, last_window_end=2.5, n_win=10, s_pick_uncertainty=0.0509, mode='swspy', 
                                                         plot_results=False)

In [ ]:
if 'results_swspy' in locals() and results_swspy:
    # Create the rose plot
    fig, ax = plot_fast_direction_rose(
        results_swspy,
        title=f"Fast Direction Rose Plot - Station AXAS2, SWSPy - Baillard Convention",
        nbins=18,  # 10° binsd
        color='steelblue',
        figsize=(10, 10)
    )
    plt.show()

In [ ]:
# Create the rose plot
#fig, ax = plot_fast_direction_rose(
#    results_baillard,
#    title=f"Fast Direction Rose Plot - Station AXAS2, Baillard",
#    nbins=18,  # 10° bins
#    color='steelblue',
#    figsize=(10, 10)
#)
#plt.show()

In [ ]:
# Create a new results dictionary with converted phi values for SWSPy results
results_swspy_converted = {}
for event_id, result in results_swspy.items():
    phi_swspy = result['result']['phi']
    phi_converted = convert_phi_baillard(phi_swspy)
    
    # Create a new result dict with converted phi but same dt and metadata
    converted_result = result.copy()
    converted_result['result'] = converted_result['result'].copy()
    converted_result['result']['phi'] = phi_converted  # Update phi to Baillard convention
    
    results_swspy_converted[event_id] = converted_result

In [ ]:
fig, axes, df = plot_splitting_timeseries_smooth(
    results_swspy,
    qc_metrics_df,
    station='AXAS2, SWSPy',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    y_width_dt=2,   # 2 samples
    sigma=2.0
)
plt.show()

In [ ]:
#fig, axes, df = plot_splitting_timeseries_smooth(
#    results_baillard,
#    qc_metrics_df,
#    station='AXAS2, Baillard',
#    x_overlap= 0.9,
#    y_width_phi=4.5,  # 5 degrees converted to radians internally
#    y_width_dt=2,   # 2 samples
#    sigma=2.0
#)  # Increase for more smoothing
#plt.show()

In [ ]:
fig, axes, df = plot_splitting_timeseries_smooth_before_after(
    results_swspy,
    qc_metrics_df,
    station='AXAS2, SWSPy',
    y_width_phi = 180 /20,  # 5 degrees converted to radians internally
    x_overlap = 0.95,
    y_width_dt=2,   # 2 samples
    sigma=2.0,
    eruption_time='2015-04-24T05:00:00Z'
)
plt.show()

In [ ]:
# Plot Baillard results around eruption
#eruption_time = pd.to_datetime("2015-04-24T05:00:00", utc=True)
#fig, axes, df_eruption_baillard = plot_splitting_timeseries_around_eruption(
#    results_baillard,
#    qc_metrics_df,
#    eruption_time=eruption_time,
#    station='AXAS2, Baillard',
#    figsize=(4, 10),
#    hours_before=24,
#    hours_after=24,
#    x_width_hours=2,
#    y_width_phi=5,
#    y_width_dt=2,
#    sigma=2
#)
#plt.show()

In [ ]:
# Plot SWSPy results around eruption
eruption_time = pd.to_datetime("2015-04-24T05:00:00", utc=True)
fig, axes, df_eruption_swpsy = plot_splitting_timeseries_around_eruption(
    results_swspy,
    qc_metrics_df,
    eruption_time=eruption_time,
    station='AXAS2, SWPSY',
    figsize=(4, 10),
    hours_before=24,
    hours_after=24,
    x_width_hours=2,
    y_width_phi=5,
    y_width_dt=2,
    sigma=2
)
plt.show()

In [ ]:

eruption_time = pd.to_datetime("2015-04-24T05:00:00", utc=True)

# Plot for SWSPy results
fig_swspy, axes_swspy, df_swspy = plot_splitting_scatter_around_eruption(
    results_swspy,
    qc_metrics_df,
    eruption_time=eruption_time,
    station='AXAS2 (SWSPy)',
    hours_before=24,
    hours_after=24,
    window_hours=3,  # 3-hour rolling average
    marker_size=40,
    marker_alpha=0.5
)

# Plot for Baillard results
#fig_baillard, axes_baillard, df_baillard = plot_splitting_scatter_around_eruption(
#    results_baillard,
#    qc_metrics_df,
#    eruption_time=eruption_time,
#    station='AXAS2 (Baillard)',
#    hours_before=24,
#    hours_after=24,
#    window_hours=3,
#    marker_size=40,
#    marker_alpha=0.5
#)
